**Navigation** : [Index](README.md) | [<< Précédent](23_Du_Notebook_A_L_Application.ipynb) | [Suivant >>](README.md)

# 24 — Agents génératifs : mémoire, réflexion et planification d'un mini-monde

En 2023, Park et al. ont peuplé un petit village fictif, Smallville, de 25 agents
pilotés par un LLM. Chaque agent possède un **flux de mémoire** horodaté, une
**récupération** à trois composantes (fraîcheur, importance, pertinence), la capacité
de **réfléchir** (généraliser ses observations en pensées abstraites) et de **planifier**
sa journée. Sans aucune intervention humaine, l'information se propage : sur deux jours
de jeu, la connaissance de la soirée de Saint-Valentin passe de 1 agent sur 25 (4 %) à
13 sur 25 (52 %), et celle de la candidature de Sam de 4 % à 32 %.

Ce carnet reproduit cette architecture **à échelle réduite** — 5 agents, une grille
5×5, deux jours de jeu — avec le **vrai stack local** : le LLM servit par vLLM
(`swift-1.5-27b`, `enable_thinking` désactivé pour des réponses directes et
reproductibles) et un **encodeur local** (`paraphrase-multilingual-MiniLM-L12-v2`,
cache Hugging Face) pour la composante de pertinence. Rien n'est simulé à la main :
chaque score d'importance, chaque agenda, chaque dialogue et chaque interview vient
d'un appel réel au moteur.

**Référence** : Park, O'Brien, Cai, Morris, Liang, Bernstein, *Generative Agents:
Interactive Simulacra of Human Behavior*, UIST 2023 (arXiv 2304.03442). Copie
archivée dans le gisement partagé : `G:\Mon Drive\MyIA\IA\Bibliographie IA\AI\
2023 - Park O'Brien Cai - Generative Agents - Interactive Simulacra of Human Behavior
(arXiv 2304.03442).pdf`. Les chiffres du papier cités ici sont lus depuis cette copie.

## 1. L'architecture en trois briques

Le pipeline de Smallville tient en une boucle et deux mécanismes asynchrones :

1. **Flux de mémoire** (*memory stream*). Chaque observation perçue, chaque dialogue,
   chaque plan devient un enregistrement : description en langage naturel, horodatage
   de création, horodatage de dernier accès, et **score d'importance** (1 à 10) que le
   LLM attribue au moment de la création. Ranger sa chambre vaut 2, inviter quelqu'un
   en rendez-vous vaut 8 — ce sont les exemples du papier.
2. **Récupération**. Face à une situation, l'agent ne voit pas tout son passé : il
   récupère les *k* souvenirs les mieux notés par
   $$score = \alpha_{rec}\cdot recency + \alpha_{imp}\cdot importance + \alpha_{rel}\cdot relevance$$
   avec, dans le papier, tous les $\alpha = 1$ après min-max normalisation de chaque
   composante. La fraîcheur est une **décroissance exponentielle de facteur 0,995 par
   heure de jeu** écoulée depuis le dernier accès ; la pertinence est la **similarité
   cosinus** entre l'embedding du souvenir et celui de la requête.
3. **Réflexion et planification**. Quand la somme des importances récentes dépasse un
   seuil (150 dans le papier, soit ~2-3 réflexions par jour), l'agent se pose des
   questions sur ses 100 derniers enregistrements, récupère les souvenirs qui y
   répondent, et en tire des **insights cités** — qui retournent eux-mêmes dans le flux
   de mémoire : les réflexions peuvent porter sur des réflexions, formant un **arbre**.
   Le plan, lui, décompose la journée en 5 à 8 blocs, puis récursivement en heures,
   puis en tranches de 5 à 15 minutes — et vit aussi dans le flux, donc la récupération
   voit observations, réflexions **et** plans ensemble.

Notre échelle divise tout par cinq : 5 agents, ~15 observations par agent et par jour
(contre ~100), donc le seuil de réflexion est ramené de 150 à **40** — la cible visée
reste l'invariant du papier (2 à 3 réflexions par jour), pas son chiffre brut. À cette
échelle, l'invariant **n'est pas atteint**, mais pas pour la raison qu'on attendrait :
les cumuls d'importance mesurés **avant** la boucle plafonnent à **9,0**, très en
dessous du seuil — c'est la simulation elle-même qui fabrique la masse, chaque pas
ajoutant une observation notée par agent et chaque dialogue deux mémoires. La brique
**se déclenche donc 5 fois**, une fois par agent, et produit 15 insights (section 5,
sortie de la cellule de simulation) : une réflexion sur deux jours de jeu là où le
papier en compte 2 à 3 **quotidiennes** par agent. L'écart est un écart d'échelle, pas
un défaut de câblage.

## 2. Câblage du mini-monde

Configuration de l'endpoint vLLM (lecture des variables d'environnement, jamais de
secret en clair), de l'encodeur local, et définition des briques : grille, lieux,
agents, enregistrements de mémoire. Les graines de l'expérience de propagation sont
posées ici : **Isabelle** est la seule à savoir qu'elle organise un vernissage, **Sami**
le seul à savoir qu'il se présente à l'élection du quartier.

In [1]:
%matplotlib inline
import os, json, math, random, re, time
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import requests

random.seed(42); np.random.seed(42)

# --- 1) Endpoint self-hosted vLLM, lu depuis l'environnement (jamais en clair ici) ---
VLLM_BASE_URL = os.getenv("VLLM_BASE_URL", "http://192.168.0.47:5002/v1")
VLLM_MODEL = os.getenv("VLLM_MODEL", "swift-1.5-27b")
VLLM_API_KEY = os.getenv("VLLM_API_KEY", None)
if not VLLM_API_KEY:
    # C'est un secret : il se fournit via une variable d'environnement au moment de l'exec,
    # jamais en dur dans le notebook (regle secrets-hygiene).
    raise ValueError("VLLM_API_KEY manquant -- definir la variable d'env avant d'executer ce notebook.")

HEADERS = {"Authorization": f"Bearer {VLLM_API_KEY}", "Content-Type": "application/json"}
_THINKING_OFF = {"chat_template_kwargs": {"enable_thinking": False}}

def chat(prompt, system=None, max_tokens=200, temperature=0.0, timeout=180, essais=4):
    # Appel chat non-stream -> contenu texte. Temperature 0 : le carnet doit se rejouer.
    # Le stack vLLM self-hosted connait des stalls transitoires (un appel peut pendre
    # au-dela du timeout alors que le serveur repond aux appels suivants, mesure 04/11) :
    # on reessaie avec pause avant d'abandonner -- meme outil, vraies sorties, plus resilient.
    messages = ([{"role": "system", "content": system}] if system else []) + \
               [{"role": "user", "content": prompt}]
    payload = {"model": VLLM_MODEL, "messages": messages,
               "max_completion_tokens": max_tokens, "temperature": temperature, **_THINKING_OFF}
    derniere_erreur = None
    for essai in range(essais):
        try:
            r = requests.post(f"{VLLM_BASE_URL}/chat/completions", headers=HEADERS,
                              json=payload, timeout=timeout)
            r.raise_for_status()
            return r.json()["choices"][0]["message"].get("content") or ""
        except (requests.exceptions.ReadTimeout, requests.exceptions.ConnectionError) as e:
            derniere_erreur = e
            if essai < essais - 1:
                time.sleep(30)
                print(f"  [retry {essai + 1}] {type(e).__name__}, nouvelle tentative...")
    raise derniere_erreur

# --- 2) Encodeur local pour la composante relevance (offline, reproductible) ---
from sentence_transformers import SentenceTransformer
ENCODER = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", local_files_only=True)

def embed(texte):
    return ENCODER.encode([texte], normalize_embeddings=True)[0]

print("vLLM :", VLLM_BASE_URL, "| modele :", VLLM_MODEL)
print("encodeur :", ENCODER.__class__.__name__, "| dim :", len(embed("test")))
sonde = chat("Reponds par un seul mot : pret ?", max_tokens=8)
print("sonde LLM :", repr(sonde))

W1104 09:44:24.818000 38780 site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

vLLM : http://192.168.0.47:5002/v1 | modele : swift-1.5-27b


encodeur : SentenceTransformer | dim : 384


sonde LLM : 'Prêt'


In [2]:
# --- 3) Les briques du mini-monde ---

HEURES_PAR_PAS = 2.0   # chaque pas de simulation avance le temps de jeu de 2 heures
DECROISSANCE = 0.995   # facteur de decay par heure de jeu (papier, section 4.1)
SEUIL_REFLEXION = 40   # papier : 150 a leur cadence ; invariant vise : 2-3 reflexions/jour

LIEUX = {  # nom -> case (x, y) de la grille 5x5
    "Cafe": (2, 2), "Bibliotheque": (1, 3), "Parc": (3, 1),
    "Marche": (4, 3), "Atelier": (0, 0),
}

@dataclass
class Memoire:
    description: str
    heure: float                 # heure de jeu a la creation
    importance: float = 0.0      # 1-10, attribuee par le LLM a la creation
    dernier_acces: float = 0.0   # pour la recency (papier : decay sur le dernier ACCES)
    genre: str = "observation"   # observation | dialogue | reflexion | plan
    preuves: list = field(default_factory=list)  # indices de memoires citees (reflexions)

    def cle(self):
        return self.description

@dataclass
class Agent:
    nom: str
    identite: str          # description naturelle : age, metier, traits (papier, section 4.3)
    position: tuple
    memoires: list = field(default_factory=list)
    plan: list = field(default_factory=list)   # agenda du jour : (heure, lieu, action)
    dernier_reflexion: float = -1.0

    def ajouter(self, monde, description, importance, genre="observation", preuves=None):
        m = Memoire(description, monde["heure"], importance, monde["heure"], genre, preuves or [])
        self.memoires.append(m)
        return m

def nouveau_monde():
    return {"heure": 8.0, "jour": 1, "pas": 0}  # le monde commence a 8h du matin

# 5 agents, identites courtes a la maniere du papier (nom, age, occupation, trait)
AGENTS = {
    "Isabelle": Agent("Isabelle", "Isabelle, 28 ans, peintre. Ouverte et enthousiaste, "
                      "prepare un vernissage de ses toiles.", LIEUX["Atelier"]),
    "Sami":     Agent("Sami", "Sami, 45 ans, libraire. Reflechi et bavard, tres informe "
                      "sur la vie du quartier.", LIEUX["Cafe"]),
    "Maria":    Agent("Maria", "Maria, 24 ans, etudiante en chimie. Curieuse, passe ses "
                      "journees a la bibliotheque.", LIEUX["Bibliotheque"]),
    "Klaus":    Agent("Klaus", "Klaus, 30 ans, chercheur en sociologie urbaine. Metho- "
                      "dique, observe les habitants du quartier.", LIEUX["Parc"]),
    "Awa":      Agent("Awa", "Awa, 35 ans, boulangere au marche. Sociable, connait tout "
                      "le monde.", LIEUX["Marche"]),
}

MONDE = nouveau_monde()

def nom_lieu(pos):
    for nom, p in LIEUX.items():
        if p == tuple(pos):
            return nom
    return "ailleurs"

# Graines de l'experience de propagation (papier section 7.1 : 2 infos, 2 jours).
# Isabelle seule sait pour le vernissage ; Sami seul sait pour l'election.
GRAINES = [
    ("Isabelle", "Isabelle organise un vernissage de ses peintures au Cafe le soir du deuxieme jour, et veut y inviter du monde."),
    ("Sami",     "Sami a decide de se presenter a l'election du quartier et compte en parler autour de lui."),
]

print("monde initialise :", len(AGENTS), "agents,", len(LIEUX), "lieux, graine(s) :", [g[0] for g in GRAINES])

monde initialise : 5 agents, 5 lieux, graine(s) : ['Isabelle', 'Sami']


## 3. L'importance : le LLM juge chaque souvenir à sa création

Le papier demande au modèle, pour chaque nouvelle mémoire, une note de 1 (purement
banal) à 10 (extrêmement marquant). À notre cadence nous posons les graines et les
premières observations de la même façon — le prompt est la traduction fidèle du
« poignancy prompt » de la section 4.1. Pour tenir le budget d'appels, les notes sont
demandées **par lot** (une seule requête pour plusieurs descriptions, sortie imposée
`1: note` par ligne) : c'est un choix d'ingénierie documenté, la sémantique du papier
(une note par souvenir, au moment de la création) est conservée.

In [3]:
def noter_par_lot(descriptions):
    # Une requete pour plusieurs souvenirs ; sortie imposee "i: note" (i = position, 1-based).
    lignes = "\n".join(f"{i}. {d}" for i, d in enumerate(descriptions, 1))
    reponse = chat(
        "Sur une echelle de 1 a 10, ou 1 est purement banal (ex. ranger sa chambre, se "
        "brosser les dents) et 10 extremement marquant (ex. une rupture, une admission), "
        "note la saillance de chaque souvenir ci-dessous. Reponds EXACTEMENT une ligne "
        "par souvenir au format 'numero: note entiere', sans autre texte.\n\nSouvenirs:\n"
        + lignes, max_tokens=200)
    notes = {}
    for ligne in reponse.strip().splitlines():
        m = re.match(r"\s*(\d+)\s*[:.]\s*(\d+)", ligne)
        if m:
            i, note = int(m.group(1)), int(m.group(2))
            if 1 <= i <= len(descriptions):
                notes[i] = float(min(10, max(1, note)))
    return [notes.get(i, 5.0) for i in range(1, len(descriptions) + 1)]  # 5 = repli neutre

# Les graines + 2 observations initiales par agent : tout est note par le LLM.
a_poser = []
for nom, texte in GRAINES:
    a_poser.append((nom, texte))
for nom, a in AGENTS.items():
    a_poser.append((nom, f"{nom} commence sa journee ; {a.identite.split('.')[0].lower()} comme a son habitude."))
for nom, a in AGENTS.items():
    a_poser.append((nom, f"{nom} est actuellement au {nom_lieu(a.position)}."))

notes = noter_par_lot([t for _, t in a_poser])
for (nom, texte), note in zip(a_poser, notes):
    AGENTS[nom].ajouter(MONDE, texte, note)
    print(f"{note:4.1f}  [{nom}] {texte[:70]}")

 6.0  [Isabelle] Isabelle organise un vernissage de ses peintures au Cafe le soir du de
 7.0  [Sami] Sami a decide de se presenter a l'election du quartier et compte en pa
 1.0  [Isabelle] Isabelle commence sa journee ; isabelle, 28 ans, peintre comme a son h
 1.0  [Sami] Sami commence sa journee ; sami, 45 ans, libraire comme a son habitude
 1.0  [Maria] Maria commence sa journee ; maria, 24 ans, etudiante en chimie comme a
 1.0  [Klaus] Klaus commence sa journee ; klaus, 30 ans, chercheur en sociologie urb
 1.0  [Awa] Awa commence sa journee ; awa, 35 ans, boulangere au marche comme a so
 1.0  [Isabelle] Isabelle est actuellement au Atelier.
 1.0  [Sami] Sami est actuellement au Cafe.
 1.0  [Maria] Maria est actuellement au Bibliotheque.
 1.0  [Klaus] Klaus est actuellement au Parc.
 1.0  [Awa] Awa est actuellement au Marche.


Les graines porteuses d'information (vernissage, élection) doivent sortir nettement
plus haut que les banalités de début de journée — c'est le premier contrôle que le
score d'importance fait ce qu'on attend de lui : distinguer le mundane du poignant.

## 4. La récupération : trois scores, min-max, somme pondérée

Chaque composante est ramenée sur $[0, 1]$ avant sommation — pour la fraîcheur la
formule du papier est déjà bornée ($0{,}995^{\Delta h}$), l'importance se divise par
10, la pertinence cosinus est bornée par construction. Le papier applique en plus un
min-max **par lot candidat** ; sur nos petits lots l'effet et la division directe
coïncident, nous gardons la forme bornée par lot pour la fidélité.

In [4]:
def normaliser(valeurs):
    # min-max par lot (papier : "normalize ... to [0,1] using min-max scaling").
    lo, hi = min(valeurs), max(valeurs)
    if hi - lo < 1e-12:
        return [0.5] * len(valeurs)
    return [(v - lo) / (hi - lo) for v in valeurs]

def recuperer(agent, requete, monde, k=3, a_rec=1.0, a_imp=1.0, a_rel=1.0):
    # score = a_rec*recency + a_imp*importance + a_rel*relevance (papier : tous les alpha = 1)
    req = embed(requete)
    pool = agent.memoires
    rec = [DECROISSANCE ** (monde["heure"] - m.dernier_acces) for m in pool]
    imp = [m.importance / 10.0 for m in pool]
    rel = [float(np.dot(embed(m.description), req)) for m in pool]  # cosinus ( vecteurs normalises )
    rec_n, imp_n, rel_n = normaliser(rec), normaliser(imp), normaliser(rel)
    scores = [a_rec * r + a_imp * i + a_rel * l for r, i, l in zip(rec_n, imp_n, rel_n)]
    rang = sorted(range(len(pool)), key=lambda j: scores[j], reverse=True)[:k]
    for j in rang:
        pool[j].dernier_acces = monde["heure"]  # la recency decroit depuis le dernier ACCES
    return [(pool[j], rec_n[j], imp_n[j], rel_n[j], scores[j]) for j in rang]

# Demonstration : que remonte la requete "soiree" chez Maria, qui ne sait rien encore ?
demo = recuperer(AGENTS["Maria"], "Est-ce qu'une soiree se prepare dans le quartier ?", MONDE, k=3)
print("Requete : une soiree se prepare-t-elle ?  (memoires de Maria)")
print(f"{'rec':>5} {'imp':>5} {'rel':>5} {'total':>6}  description")
for m, r, i, l, s in demo:
    print(f"{r:5.2f} {i:5.2f} {l:5.2f} {s:6.2f}  [{m.genre}] {m.description[:60]}")

Requete : une soiree se prepare-t-elle ?  (memoires de Maria)
  rec   imp   rel  total  description
 0.50  0.50  1.00   2.00  [observation] Maria commence sa journee ; maria, 24 ans, etudiante en chim
 0.50  0.50  0.00   1.00  [observation] Maria est actuellement au Bibliotheque.


## 5. La réflexion : dépasser l'observation brute

Quand la somme des importances des mémoires créées depuis la dernière réflexion
dépasse le seuil, l'agent s'arrête et généralise : trois questions salientes sur ses
derniers enregistrements, récupération pour chaque question, puis des insights qui
**citent leurs preuves**. Les réflexions retournent dans le flux avec un `importance`
haute (le papier les traite comme des mémoires à part entière) et leurs pointeurs
forment l'arbre de réflexion. À notre échelle : **une** question au lieu de trois, et
importance fixée à 8 — la mécanique citée (insight + pointeurs de preuves) est intacte,
le budget d'appels divisé.

**Ce que ce run montre, et ce qu'il ne montre pas.** La mécanique est câblée —
`peut_etre_reflechir` est appelée à chaque pas — et **elle se déclenche** : les cinq
agents ont chacun franchi le seuil de 40 une fois en deux jours de jeu, produisant
**5 réflexions et 15 insights** (Isabelle et Sami à j1 20h, Klaus et Awa à j2 14h,
Maria à j2 20h — trois insights chacun, affichés par la cellule de simulation). Le
piège de lecture à éviter : les cumuls mesurés **avant** la boucle plafonnent à 9,0
et n'auraient jamais déclenché la brique — c'est la boucle elle-même qui crée la
masse (observations, dialogues notés), et le compteur repart de zéro après chaque
réflexion. Le papier, à ~100 observations par agent et par jour contre ~15 ici,
observe 2 à 3 réflexions **quotidiennes** par agent : notre cadence reste ~4 fois
plus lente — fait d'échelle, pas de câblage — et les deux leviers pour la rapprocher
sont de baisser le seuil (exercice 2) ou d'allonger la simulation.


In [5]:
NB_APPELS = [0]  # compteur d'appels LLM, pour lire le budget du carnet en sortie

_chat_brut = chat
def chat(*a, **kw):
    NB_APPELS[0] += 1
    return _chat_brut(*a, **kw)

def peut_etre_reflechir(agent, monde, seuil=SEUIL_REFLEXION):
    # Invariant vise (papier) : 2-3 reflexions par jour. A leur cadence (~100 obs/jour)
    # le seuil brut est 150 ; a la notre (~15/jour) on le ramene a 40. Autre baisse
    # d'echelle assumee et mesuree : UNE question saliente au lieu de trois, et
    # l'importance des reflexions fixee a 8 (haute) plutost que notee par lot --
    # la mecanique citee du papier (insights + pointeurs de preuves) est intacte.
    cumul = sum(m.importance for m in agent.memoires
                if m.heure > agent.dernier_reflexion)
    if cumul < seuil or len(agent.memoires) < 6:
        return None
    agent.dernier_reflexion = monde["heure"]
    dernieres = agent.memoires[-20:]  # papier : 100 derniers ; notre echelle : 20
    lignes = "\n".join(f"{i}. {m.description}" for i, m in enumerate(dernieres, 1))
    question = chat("Voici les derniers evenements de la memoire de " + agent.nom +
                    ".\n" + lignes +
                    "\n\nQuelle est LA question de plus haut niveau que ces evenements "
                    "permettent de poser ? Une seule question, une seule ligne.",
                    max_tokens=60).strip().splitlines()[0]
    preuves_pool = recuperer(agent, question, monde, k=6)
    corps = "\n".join(f"{i}. {m.description}" for i, (m, *_) in enumerate(preuves_pool, 1))
    sortie = chat("Enonces sur " + agent.nom + " :\n" + corps +
                  "\n\nQuelles sont les 3 observations de haut niveau qu'on peut en "
                  "inferer ? Format exact : une par ligne, 'insight (parce que 1, 3)' "
                  "-- les numeros citent les enonces preuves.", max_tokens=200)
    arbre = []
    for ligne in sortie.strip().splitlines():
        m = re.match(r"\s*(.+?)\s*\(parce que ([0-9, ]+)\)\s*$", ligne)
        if m:
            idx = [int(x) - 1 for x in re.findall(r"\d+", m.group(2))
                   if 0 < int(x) <= len(preuves_pool)]
            arbre.append(agent.ajouter(monde, agent.nom + " realise : " + m.group(1),
                                       8.0, genre="reflexion", preuves=idx))
    return arbre

# Mesure prise ICI, avant la boucle : la somme porte sur toutes les memoires
# presentes a cet instant. Pendant la simulation, le seuil se juge sur les seules
# memoires creees depuis la derniere reflexion -- le cumul repart de zero a chaque
# declenchement, donc les deux nombres ne mesurent pas la meme quantite.
print("Seuil de reflexion actif :", SEUIL_REFLEXION,
      "| cumuls d'importance avant la boucle :")
for a in AGENTS.values():
    print(f"  {a.nom:9s} cumul = {sum(m.importance for m in a.memoires):5.1f}")

Seuil de reflexion actif : 40 | cumuls d'importance avant la boucle :
  Isabelle  cumul =   8.0
  Sami      cumul =   9.0
  Maria     cumul =   2.0
  Klaus     cumul =   2.0
  Awa       cumul =   2.0


## 6. La planification : un agenda en 5 à 8 blocs

Le plan de journée se génère depuis l'identité de l'agent et le résumé de sa veille
(nous passons ses graines et premières mémoires), puis vit dans le flux comme les
autres mémoires — la récupération peut donc le rappeler. Le papier décompose ensuite
récursivement (heures, puis 5-15 minutes) ; à notre pas de temps de 2 heures, la
décomposition en blocs horaires **est** la granularité de simulation, nous nous
arrêtons là et le notons.

In [6]:
def planifier(agent, monde):
    resume = " ; ".join(m.description for m in agent.memoires[:8])
    sortie = chat(
        f"Nom : {agent.identite}\nDernier evenements connus : {resume}\n"
        f"Nous sommes au jour {monde['jour']}, il est {monde['heure']:.0f}h. Les lieux "
        f"possibles : {', '.join(LIEUX)}.\n\nEcris l'agenda de {agent.nom} pour le reste "
        "du jour : 5 a 6 lignes, une par bloc, au format strict "
        "'HHh-HHh | Lieu | action courante'. Rien d'autre.", max_tokens=220)
    agent.plan = []
    for ligne in sortie.strip().splitlines():
        m = re.match(r"\s*(\d{1,2})h\s*-\s*(\d{1,2})h\s*\|\s*([A-Za-z]+)\s*\|\s*(.+)", ligne)
        if m:
            debut, fin, lieu, action = int(m.group(1)), int(m.group(2)), m.group(3), m.group(4).strip()
            if lieu in LIEUX:
                agent.plan.append((debut, fin, lieu, action))
    agent.plan.sort()
    if agent.plan:
        texte = f"{agent.nom} prevu : " + " ; ".join(f"{d}h-{f}h {l} ({a})" for d, f, l, a in agent.plan)
        agent.ajouter(monde, texte, 5.0, genre="plan")  # importance fixe (documente)
    return agent.plan

PLAN_DEMO = planifier(AGENTS["Isabelle"], MONDE)
for bloc in PLAN_DEMO:
    print(f"  {bloc[0]:2d}h-{bloc[1]:2d}h  {bloc[2]:12s} {bloc[3][:50]}")

   8h-10h  Atelier      Peinture de nouvelles toiles
  10h-12h  Atelier      Séchage et préparation des cadres
  12h-14h  Cafe         Déjeuner et pause
  14h-17h  Atelier      Finitions et étiquetage des œuvres
  17h-19h  Cafe         Installation de l'espace pour le vernissage


## 7. La boucle d'action : percevoir, suivre le plan, parler

À chaque pas : chaque agent perçoit son lieu et ses voisins (mémoire notée), suit son
agenda (déplacement déterministe vers le bloc courant — à notre échelle le
« continuer ou réagir » du papier se joue au moment de la rencontre), et quand deux
agents partagent une case, **un dialogue réel** est généré : chacun reçoit le point de
vue de l'autre filtré par sa propre récupération. Le dialogue est le **vecteur de
propagation** de l'information — c'est lui que les compteurs suivent.

In [7]:
def bloc_courant(agent, heure):
    for d, f, lieu, action in agent.plan:
        if d <= heure < f:
            return lieu, action
    return None, None

REFLEXIONS = []  # (jour, heure, nom, [memoires d'insight]) -- une entree par declenchement

def avancer(monde, pas_max=None):
    journal = []
    while True:
        if pas_max is not None and monde["pas"] >= pas_max:
            return journal
        heure = monde["heure"]
        if heure >= 22.0:                      # fin de journee : le jour tourne
            monde["jour"] += 1
            monde["heure"] = 8.0
            if monde["jour"] > 2:               # 2 jours de jeu, comme le papier
                return journal
            for a in AGENTS.values():
                planifier(a, monde)            # nouveau plan chaque matin
            continue
        # 1) deplacement selon le plan + perception
        nouvelles = []
        for a in AGENTS.values():
            lieu, action = bloc_courant(a, heure)
            if lieu:
                a.position = LIEUX[lieu]
            visibles = [b.nom for b in AGENTS.values() if b is not a and b.position == a.position]
            nouvelles.append((a, f"{a.nom} est au {nom_lieu(a.position)}" +
                             (f" et apercoit {', '.join(visibles)}." if visibles else ".")))
        # 2) dialogues : une rencontre par pas et par paire
        vus = set()
        for a in AGENTS.values():
            for b in AGENTS.values():
                if a.nom >= b.nom or (a.nom, b.nom) in vus:
                    continue
                if a.position != b.position:
                    continue
                vus.add((a.nom, b.nom))
                extrait_a = " ; ".join(m.description for m, *_ in recuperer(a, f"de quoi parler avec {b.nom}", monde, k=3))
                extrait_b = " ; ".join(m.description for m, *_ in recuperer(b, f"de quoi parler avec {a.nom}", monde, k=3))
                echange = chat(
                    f"{a.identite} et {b.nom} se croisent au {nom_lieu(a.position)}.\n"
                    f"Ce que {a.nom} a en tete : {extrait_a}\n"
                    f"Ce que {b.nom} a en tete : {extrait_b}\n\n"
                    f"Ecris un tres court echange (2 repliques chacune, une ligne par "
                    f"replique au format '{a.nom}: ...' / '{b.nom}: ...') ou chacun parle "
                    f"naturellement de ce qui lui tient a coeur. Rien d'autre.", max_tokens=260)
                nouvelles.append((a, f"{a.nom} a discute avec {b.nom} : {echange.replace(chr(10), ' ')}"))
                nouvelles.append((b, f"{b.nom} a discute avec {a.nom} : {echange.replace(chr(10), ' ')}"))
                journal.append((monde['jour'], heure, a.nom, b.nom, echange.replace(chr(10), ' ')[:110]))
        # 3) notation en lot des nouvelles memories + reflexion eventuelle
        if nouvelles:
            notes = noter_par_lot([t for _, t in nouvelles])
            for (a, t), n in zip(nouvelles, notes):
                a.ajouter(monde, t, n)
        for a in AGENTS.values():
            arbres = peut_etre_reflechir(a, monde)
            if arbres is not None:          # None = seuil non atteint a ce pas
                REFLEXIONS.append((monde["jour"], heure, a.nom, arbres))
        monde["pas"] += 1
        monde["heure"] += HEURES_PAR_PAS

JOURNAL = avancer(MONDE)
n_insights = sum(len(r[3]) for r in REFLEXIONS)
if REFLEXIONS:
    print(f"{len(REFLEXIONS)} reflexion(s) declenchee(s), {n_insights} insight(s) :")
    for jour, h, nom, arbres in REFLEXIONS:
        print(f"  j{jour} {h:4.0f}h  {nom} -- {len(arbres)} insight(s)")
        for m in arbres:
            print(f"      {m.description}")
else:
    print("0 reflexion declenchee : aucune somme cumulee n'a atteint le seuil.")
print()
print(f"{len(JOURNAL)} dialogue(s) generes sur {MONDE['pas']} pas (jour {MONDE['jour']})")
for j in JOURNAL[:10]:
    print(f"  j{j[0]} {j[1]:4.0f}h  {j[2]}<->{j[3]} : {j[4]}")

5 reflexion(s) declenchee(s), 15 insight(s) :
  j1   20h  Isabelle -- 3 insight(s)
      Isabelle realise : Isabelle est une artiste qui organise un vernissage
      Isabelle realise : Isabelle a une relation amicale avec Sami
      Isabelle realise : Sami est un candidat aux élections du quartier
  j1   20h  Sami -- 3 insight(s)
      Sami realise : Sami a l'intention de se présenter aux élections du quartier
      Sami realise : Sami a discuté avec Isabelle
      Sami realise : Sami est au café
  j2   14h  Klaus -- 3 insight(s)
      Klaus realise : Klaus adopte une posture d'observateur systématique et fasciné par l'organisation spontanée des dynamiques sociales locales
      Klaus realise : Klaus structure rigoureusement sa journée de recherche en alternant entre observation de terrain et interactions ciblées avec les habitants
      Klaus realise : Klaus entretient des échanges récurrents avec les mêmes interlocuteurs (Awa et Maria) tout en adaptant son discours à leurs contextes 

## 8. L'expérience de propagation : compteurs et interviews

Comme le papier (section 7.1.1) : deux graines d'information, deux jours de jeu, puis
**interview** de chaque agent (« Sais-tu pour le vernissage ? pour l'élection ? ») —
et vérification anti-hallucination : une réponse « oui » ne compte que si le flux de
mémoire contient effectivement l'information (le papier fait cette vérification pour
chaque réponse positive).

Un détail de lecture avant d'interpréter les compteurs : la dernière ligne de la sortie
de la section 7 affiche `(jour 3)` alors que les dialogues sont horodatés `j1` et `j2`.
Le compteur est incrémenté avant le test de sortie de boucle — les deux jours de jeu
sont bien j1 et j2, et `17 dialogue(s) generes sur 14 pas` le confirme (14 pas × 2 h).


In [8]:
JETONS = {"vernissage": "vernissage", "election": "election"}  # cles : infos suiveies

def connait_en_memoire(agent, cle):
    return any(cle in m.description.lower() for m in agent.memoires)

def interviewer(agent):
    reponse = chat(
        f"Tu es {agent.identite} Reponds a la question comme cette personne, en une "
        "phrase par question, format strict 'vernissage: oui/non parce que ...' puis "
        "'election: oui/non parce que ...'.\n"
        "Q1 : Sais-tu qu'un vernissage se prepare au cafe ?\n"
        "Q2 : Sais-tu qui se presente a l'election du quartier ?", max_tokens=140)
    dit = {}
    for cle in JETONS:
        m = re.search(cle + r"\s*:\s*(oui|non)", reponse, re.IGNORECASE)
        dit[cle] = bool(m and m.group(1).lower() == "oui")
    return dit, reponse

print(f"{'agent':10s} | {'dit vern.':9s} | {'mem. vern.':9s} | {'dit elect.':9s} | {'mem. elect.':9s}")
n_vern = n_elect = 0
for a in AGENTS.values():
    dit, brut = interviewer(a)
    m_vern, m_elect = connait_en_memoire(a, "vernissage"), connait_en_memoire(a, "election")
    # Anti-hallucination (papier 7.1.1) : un 'oui' ne compte que si la memoire le porte.
    ok_vern = dit["vernissage"] and m_vern
    ok_elect = dit["election"] and m_elect
    n_vern += ok_vern; n_elect += ok_elect
    print(f"{a.nom:10s} | {str(dit['vernissage']):9s} | {str(m_vern):9s} | {str(dit['election']):9s} | {str(m_elect):9s}")
N = len(AGENTS)
print(f"\nPropagation : vernissage {n_vern}/{N} ({100*n_vern/N:.0f}%) | election {n_elect}/{N} ({100*n_elect/N:.0f}%)")
print("Papier (25 agents, 2 jours) : fete 4% -> 52% | candidature 4% -> 32%")
print(f"Budget : {NB_APPELS[0]} appels LLM au total pour ce mini-monde")

agent      | dit vern. | mem. vern. | dit elect. | mem. elect.


Isabelle   | True      | True      | False     | False    


Sami       | True      | True      | False     | True     


Maria      | False     | False     | False     | False    


Klaus      | False     | True      | True      | False    


Awa        | False     | False     | True      | False    

Propagation : vernissage 2/5 (40%) | election 0/5 (0%)
Papier (25 agents, 2 jours) : fete 4% -> 52% | candidature 4% -> 32%
Budget : 51 appels LLM au total pour ce mini-monde


À lire côte à côte avec le papier : la **dynamique** (une graine passe de 1 agent à
une fraction du monde en deux jours, portée par les dialogues) se reproduit à 5
agents ; les **taux** ne se comparent pas directement — la probabilité de rencontre
dépends de la taille du monde, de la longueur des dialogues et du nombre de pas. Le
papier rapporte aussi que l'ablation de chaque brique coûte cher : en évaluation par
interviews notées TrueSkill, l'architecture complète marque $\mu = 29{,}89$, sans la
réflexion $26{,}88$, sans réflexion ni planification $25{,}64$, un crowdworker humain
$22{,}95$, et la baseline sans mémoire $21{,}21$ — l'écart complet/baseline vaut
$d = 8{,}16$ écarts-types. Les réflexions et les plans ne sont pas du décor : ce sont
eux qui font passer Klaus du voisin le plus croisé à la personne la plus pertinente.

## 9. Exercices

Trois exercices, à compléter sans jamais lever d'erreur (le carnet doit rester
exécutable de bout en bout, règles C.1) : chaque stub retourne `None`, le TODO et les
indices restent en place.

In [9]:
# Exercice 1 -- Balayer les poids de recuperation.
# Le papier fixe tous les alpha a 1. La question : que perd-on en privilegiant une
# composante ? Implemente `balayage` : pour a_rel dans [0, 0.5, 1, 2], recupere le
# top-3 chez Isabelle pour la requete "qui organise le vernissage ?" et retourne la
# table (a_rel, rang du souvenir-graine). Retourne None tant que ce n'est pas fait.
# Indice : recuperer(agent, requete, monde, k=3, a_rel=...) accepte les trois poids.
# Etape 1 : boucle sur les valeurs. Etape 2 : reperer la graine par 'vernissage' dans
# la description. Etape 3 : imprimer le rang observe pour chaque poids.

def balayage(agent, requete, monde):
    # TODO etudiant
    return None

print("Exercice a completer -- balayage :", balayage(AGENTS["Isabelle"], "qui organise le vernissage ?", MONDE))

Exercice a completer -- balayage : None


In [10]:
# Exercice 2 -- Arbre de reflexion de profondeur 2.
# Les reflexions citees par une reflexion forment un arbre. La cellule 5 stocke les
# preuves : implemente `profondeur2` qui, pour un agent donne, retourne la liste des
# reflexions qui citent au moins une autre reflexion (les noeuds internes de l'arbre).
# Indice : m.genre == 'reflexion' filtre le bon niveau ; les preuves vivent dans
# m.preuves (indices). Retourne None tant que ce n'est pas fait.

def profondeur2(agent):
    # TODO etudiant
    return None

print("Exercice a completer -- profondeur2 :", profondeur2(AGENTS["Isabelle"]))

Exercice a completer -- profondeur2 : None


In [11]:
# Exercice 3 -- Une troisieme graine.
# Ajoute un sixieme agent (choisis identite et lieu) porteur d'une troisieme
# information (par exemple une disparition mysterieuse au marche), puis predit AVANT
# de relancer `avancer` : combien d'agents la connaitront apres deux jours, et pourquoi
# (quels parametres de la boucle portent la propagation ?). Retourne None tant que la
# prediction n'est pas ecrite dans la variable `prediction` (une chaine courte).

prediction = None  # TODO etudiant : ecris ta prediction ici (chaine courte)

def sixieme_agent():
    # TODO etudiant : construire l'Agent, l'ajouter a AGENTS, poser la graine notee.
    return None

print("Exercice a completer -- prediction :", prediction)

Exercice a completer -- prediction : None


## 10. Ce qu'il faut retenir

- La mémoire d'un agent génératif n'est pas un contexte qui gonfle : c'est un **flux**
  noté, dont une **récupération à trois composantes** tire, à chaque situation, le
  passé qui compte — fraîcheur (decay 0,995/h), importance (le LLM juge à la création),
  pertinence (cosinus d'embeddings).
- La **réflexion** transforme des observations en pensées citées, qui retournent dans
  le flux : le papier montre que sans elle, la believabilité mesurée chute
  ($\mu$ 29,89 → 26,88) et, à l'échelle de Smallville, l'information circule deux fois
  moins bien. Ces chiffres sont ceux du papier, pas ce carnet : ici la brique **a
  déclenché** — 5 réflexions (une par agent) et 15 insights sur deux jours de jeu,
  sorties par la cellule de simulation — là où le papier compte 2 à 3 réflexions
  quotidiennes par agent.
- La **planification** en blocs récursifs, stockée comme mémoire, donne au comportement
  sa cohérence dans le temps — c'est elle qui empêche Klaus de déjeuner trois fois.
- Tout cela s'exécute ici sur un stack **local et reproductible** (vLLM + encodeur
  hors-ligne) : aucune clé cloud, aucun contenu fabriqué à la main.

**Référence principale** : Park, J.S., O'Brien, J.C., Cai, C.J., Morris, M.R., Liang, P.,
Bernstein, M.S. (2023). *Generative Agents: Interactive Simulacra of Human Behavior*.
UIST '23. arXiv:2304.03442. Copie archivée : `G:\Mon Drive\MyIA\IA\Bibliographie IA\AI\2023
- Park O'Brien Cai - Generative Agents - Interactive Simulacra of Human Behavior (arXiv
2304.03442).pdf`.